In [70]:
import os
from one.api import ONE
from brainbox.io.one import SpikeSortingLoader
import pandas as pd
from tqdm import tqdm
from itertools import groupby
import numpy as np
from tqdm import tqdm

In [ ]:
one = ONE(
    base_url='https://openalyx.internationalbrainlab.org',
    password='international',
    cache_dir="./data"
)

eids_1 = one.search(project='brainwide', datasets='spikes.times.npy')
eids_2 = one.search(tag="Brainwidemap")
eids = list(set(eids_1).intersection(eids_2))
np.save("eids", np.array(eids).astype(str))

In [44]:
eids = np.load("eids.npy")

In [6]:
def download_data(eid):
    _, pnames = one.eid2pid(eid) # extract probe names
    for pname in pnames:
        sl = SpikeSortingLoader(eid=eid, pname=pname, one=one)
        sl.load_spike_sorting(good_units=True)

In [ ]:
for eid in eids:
    download_data(eid)

In [67]:
def make_session_dataset(eid, probe_name):
    # load curated units
    sl = SpikeSortingLoader(eid=eid, pname=probe_name, one=one)
    spikes, clusters, channels = sl.load_spike_sorting(good_units=True)
    clusters = sl.merge_clusters(spikes, clusters, channels)
    
    # Safety check: for standard IBL outputs cluster_id should match row index
    cluster_ids = np.asarray(clusters.cluster_id).astype(int)
    assert np.array_equal(cluster_ids, np.arange(len(cluster_ids))), \
        f"cluster_id != row index for eid={eid}, probe={probe_name}"

    # Accepted neurons must come from the filtered spikes object
    accepted_ids = np.unique(np.asarray(spikes.clusters).astype(int)) 
    
    # make neuron dataset
    data = {}
    for neuron_id in accepted_ids:
        temp = {}
        firing_rate = float(clusters.firing_rate[neuron_id])
        if firing_rate <= 0.05: continue
        
        temp["firing_rate"] = firing_rate
        temp["neuron_spike_times"] = spikes.times[spikes.clusters == neuron_id]
        temp["n_spikes"] = len(neuron_spike_times)
        temp["acronym"] = str(clusters.acronym[neuron_id])
        temp["atlas_id"] = int(clusters.atlas_id[neuron_id])
        temp["coord"] = np.array([
            clusters.x[neuron_id], 
            clusters.y[neuron_id], 
            clusters.z[neuron_id]
        ]).astype(float)
        temp["hemisphere"] = "L" if coord[0] < 0 else "R"
        temp["histology"] = sl.histology in {"resolved", "alf"}
        data[neuron_id] = temp
        
    return data

In [68]:
eids = [
    '1ec23a70-b94b-4e9c-a0df-8c2151da3761',
    '5522ac4b-0e41-4c53-836a-aaa17e82b9eb'
]

In [71]:
os.makedirs("output", exist_ok=True)

for eid in tqdm(eids):
    session_data = {
        "session_id": eid,
        "neural_data": {}
    }
        
    _, pnames = one.eid2pid(eid)
    
    for probe_name in pnames:
        probe_data = make_session_dataset(eid, probe_name)
        session_data["neural_data"][probe_name] = probe_data
        
    np.save(f"output/{eid}.npy", session_data)

100%|██████████| 2/2 [00:26<00:00, 13.26s/it]
